In [1]:
%load_ext autoreload
%autoreload 2

# IMPORTS

In [2]:
# Project setup
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [3]:
# Third-party libraries
import pandas as pd

# Scikit-learn
from sklearn.model_selection import train_test_split

from src.preprocessing import (
    prepare_data,
    prepare_target,
    TextCleaner,
    create_preprocessor
)

from src.config import RANDOM_STATE

# Load data & Split data

In [4]:
# Exit notebooks/ and access the dataset in data/
data_path = project_root / "data" / "Airline_review.csv"
raw_df = pd.read_csv(data_path)

# Split the raw data into training-validation and test sets (80/20)
train_val_df, test_df = train_test_split(
    raw_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=raw_df['Recommended']
)

# Split the training-validation data into training and validation sets (80/20)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=train_val_df['Recommended']
)



# Data preprocessing

## 1\. Dataset preparation

In [5]:
# remove unused features
cols_to_drop = [
    'Unnamed: 0',
    'Review Date',
    'Verified',
    'Route',
    'Airline Name',
    'Overall_Rating',
    'Aircraft',
    'Date Flown'
]

train_df = prepare_data(train_df, cols_to_drop)
val_df = prepare_data(val_df, cols_to_drop)
test_df = prepare_data(test_df, cols_to_drop)

## 2\. Target preparation

In [6]:
train_df = prepare_target(train_df)
val_df = prepare_target(val_df)
test_df = prepare_target(test_df)

## 3\. Feature analysis and preprocessing decisions

In [7]:
# target columns
target_col = 'Recommended_num'
target_cat_col = 'Recommended'

### 3\.1 Numeric features

In [8]:
num_cols = ['Seat Comfort', 'Cabin Staff Service', 'Food & Beverages',
            'Ground Service', 'Inflight Entertainment', 'Wifi & Connectivity',
            'Value For Money']

In [9]:
print('Missing values in Numeric columns:')
for col in num_cols:
    print(f'{col}: {train_df[col].isnull().mean() * 100:.2f}%')

Missing values in Numeric columns:
Seat Comfort: 17.91%
Cabin Staff Service: 18.36%
Food & Beverages: 37.37%
Ground Service: 20.60%
Inflight Entertainment: 53.51%
Wifi & Connectivity: 74.40%
Value For Money: 4.64%


- Missing values in service ratings are likely related to service availability rather than random data loss. Therefore, they will be imputed with a constant value (-1) during preprocessing to preserve this information.

- All rating features use the same 0–5 scale. If needed, scaling will be applied later in the preprocessing pipeline.

### 3\.2 Categorical features

In [10]:
cat_cols = ['Type Of Traveller', 'Seat Type']

In [11]:
print('Missing values in Categorical columns:')
for col in cat_cols:
    print(f'{col}: {train_df[col].isnull().mean() * 100:.2f}%')

Missing values in Categorical columns:
Type Of Traveller: 16.08%
Seat Type: 4.77%


- Missing values in categorical features will be replaced with ***Unknown***.

- The features will be encoded using `One-Hot Encoding` during preprocessing.

### 3\.3 Text features

In [12]:
text_col = 'Review_Text'

## 4\. Text preprocessing

### 4\.1 Text cleaning examples

In [13]:
text_cleaner = TextCleaner()

sample = train_df['Review_Text'].sample(5, random_state=RANDOM_STATE)

cleaned = text_cleaner.fit_transform(sample)

check_df = pd.DataFrame({
    'original_text': sample,
    'cleaned_text': cleaned
})

with pd.option_context(
        'display.max_colwidth', None,
        'display.max_columns', None
):
    display(check_df)

,original_text,cleaned_text
7666,"""compensation is due"" Gatwick to Split. The flight was more than four hours delayed - with no explanation delivered at any point - and the compensation claim process has so far yielded nothing but a case number and automated responses. Croatia Airlines’ automated customer care number simply hangs up at some point - no matter how you click your way trough it (if anyone has a magic number up their sleeve, I’d be glad to have it!). Even though compensation is due under EU law, the airline has made absolutely no effort to comply. I’m averaging three e-mails per week - all left unanswered. In short: a one hour delay with no information was followed by another hour delay at the gate (with no explanation) and another two hours in the plane - again, with no explanation. The most substantial snippet of information we got from the pilot: “We might be here for three minutes - we might be here for another 90 minutes “. Even though the staff were friendly, at no point were we offered drinks or food. We had to make our way to the back of the plane as ask for something which was still limited to a glass of water. They noticed I was pregnant, because I was shifted from the exit seat, but that didn’t really prompt the possibility of receiving anything else but that precious glass of H2O. I am not blaming the staff, I am blaming the airline. Just be wary when booking and aware that there is virtually no real customer service to be expected.","""compensation is due"" Gatwick to Split. The flight was more than four hours delayed with no explanation delivered at any point and the compensation claim process has so far yielded nothing but a case number and automated responses. Croatia Airlines' automated customer care number simply hangs up at some point no matter how you click your way trough it (if anyone has a magic number up their sleeve, I'd be glad to have it!). Even though compensation is due under EU law, the airline has made absolutely no effort to comply. I'm averaging three e-mails per week all left unanswered. In short: a one hour delay with no information was followed by another hour delay at the gate (with no explanation) and another two hours in the plane again, with no explanation. The most substantial snippet of information we got from the pilot: ""We might be here for three minutes we might be here for another 90 minutes "". Even though the staff were friendly, at no point were we offered drinks or food. We had to make our way to the back of the plane as ask for something which was still limited to a glass of water. They noticed I was pregnant, because I was shifted from the exit seat, but that didn't really prompt the possibility of receiving anything else but that precious glass of H2O. I am not blaming the staff, I am blaming the airline. Just be wary when booking and aware that there is virtually no real customer service to be expected."
18966,"""most uncomfortable flight"" It cost $100 to check a bag for the most uncomfortable flight I’ve ever been on. I would have been further ahead to ship all my stuff or just fly with another airline. Actually would have saved money I think and any other flight would have given me a bigger better seat, a snack, and a beverage just for flying with them. What a joke","""most uncomfortable flight"" It cost $100 to check a bag for the most uncomfortable flight I've ever been on. I would have been further ahead to ship all my stuff or just fly with another airline. Actually would have saved money I think and any other flight would have given me a bigger better seat, a snack, and a beverage just for flying with them. What a joke"
7145,"""not fly with them again"" Denpasar to Jakarta. I'm giving a rating of 4 is because the flight was right on time. The person at the check in counter let another passenger jump the queue, even after I told her what happen. She asked me if I want a window seat and I accept it (but instead assigned me in a seat between two people. 

### 4\.2 Preprocessing decisions

In [14]:
# checking for the presence of URLs
url_mask = train_df['Review_Text'].str.contains(
    r'https?://|www\.',
    regex=True,
    case=False,
    na=False
)

url_mask.sum()

np.int64(6)

## **Preprocessing decisions:**

- Review titles and review texts were combined into a single text feature (Review_Text). Template titles such as "<Airline> customer review" were removed as they do not provide useful information for classification.

- Numbers were preserved during text preprocessing because some numerical expressions may provide useful contextual information (e.g., delays, prices, aircraft models, or ratings). Their impact on model performance will be evaluated during model development.

- No HTML tags were found in the current dataset during exploration. Therefore, this preprocessing step does not affect the current data. However, HTML removal is included in the text cleaning pipeline to handle possible formatting artifacts in future data.

- A small number of URLs were found in the review texts (6 cases). Since links do not provide meaningful information for sentiment classification and may introduce unnecessary noise, they are removed during text preprocessing. This step also helps handle possible URLs in future data.

- Different apostrophe characters were found in the review texts. All variants were normalized to the standard apostrophe (') to ensure consistent text representation.

- Different types of quotation marks were also normalized to the standard double quote ("). This provides a consistent text representation while preserving the original content.

- Repeated punctuation (e.g., multiple exclamation or question marks) was reduced to a single symbol, and repeated emoticons were normalized. This removes formatting inconsistencies while preserving the presence of emphasis or emotion.

- Whitespace was normalized by replacing line breaks and tab characters with spaces, collapsing multiple consecutive spaces into a single space, and trimming leading and trailing whitespace. This ensures a clean and consistent text format without altering the semantic content.

- Standalone punctuation marks were removed to reduce noise from isolated symbols while preserving punctuation that may carry emotional information, such as exclamation marks, question marks, and emoticons.

- Text features were prepared by concatenating review title and review text, followed by text cleaning. TF-IDF vectorization with unigrams and bigrams is applied during preprocessing.







# 5. Preprocessing pipeline

## Preprocessing summary

The preprocessing pipeline combines three types of features:

- Numerical features:
  - missing values are replaced with -1, as 0 is a valid rating value;
  - scaling is not applied at this stage.

- Categorical features:
  - missing values are replaced with "Unknown";
  - categories are encoded using One-Hot Encoding;
  - unknown categories are ignored during transformation.

- Text features:
  - review title and review text are combined into a single text feature;
  - text is cleaned and transformed using TF-IDF with unigrams and bigrams.

In [15]:
# define input features and target
input_cols = num_cols + cat_cols + [text_col]

X_train = train_df[input_cols].copy()
X_val = val_df[input_cols].copy()
X_test = test_df[input_cols].copy()

y_train = train_df[target_col]
y_val = val_df[target_col]
y_test = test_df[target_col]

## Preprocessing validation

In [16]:
preprocessor = create_preprocessor(
    num_cols,
    cat_cols,
    text_col
)
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

X_train_processed.shape, X_val_processed.shape, X_test_processed.shape

((14828, 153906), (3708, 153906), (4635, 153906))

The fitted preprocessing pipeline was applied to validation and test sets using `transform()` only. All datasets produced the same number of features, confirming consistent feature generation.

The fitted pipeline produces a consistent feature space for training, validation, and test data.

The complete preprocessing pipeline is implemented in src/preprocessing.py and instantiated using create_preprocessor()